In [13]:
"""
Générateur de jeux de données TSPTW-D
======================================
Conforme au modèle du Livrable 1 (TSPTW-D) :
  - Graphe complet, coûts euclidiennes × scale → minutes
  - Fenêtres temporelles [a_i, b_i] cohérentes (instance réalisable)
  - Temps de service s_i
  - Perturbations dynamiques (arc, t_début, t_fin, alpha)
 
Format du fichier JSON généré
------------------------------
{
  "meta": { "n_clients": int, "scale": float, "seed": int },
  "depot": { "id": 0, "x": float, "y": float, "a": float, "b": float, "service": float },
  "clients": [
    { "id": int, "x": float, "y": float, "a": float, "b": float, "service": float },
    ...
  ],
  "perturbations": [
    { "arc": [i, j], "t_start": float, "t_end": float, "alpha": float },
    ...
  ]
}
"""
 
import json
import math
import random
from pathlib import Path
 
 
# ---------------------------------------------------------------------------
# Classe Ville (copie du projet pour pouvoir simuler la propagation)
# ---------------------------------------------------------------------------
class Ville:
    def __init__(self, nom: str, x: float, y: float,
                 fenetre: tuple = (0, float('inf')),
                 service: float = 0.0):
        self.nom     = nom
        self.x       = x
        self.y       = y
        self.a       = fenetre[0]
        self.b       = fenetre[1]
        self.service = service
 
    def __repr__(self):
        return self.nom
 
 
# ---------------------------------------------------------------------------
# Utilitaires
# ---------------------------------------------------------------------------
 
def _distance_minutes(v1: Ville, v2: Ville, scale: float) -> float:
    """Distance euclidienne convertie en minutes (c_ij^base)."""
    return math.hypot(v1.x - v2.x, v1.y - v2.y) * scale
 
 
def _propagation_greedy(villes: list[Ville], scale: float) -> float:
    """
    Simule une tournée dans l'ordre naturel (dépôt → v1 → v2 → … → dépôt)
    et retourne l'heure de retour au dépôt.
    Sert uniquement à calibrer b_0 (fenêtre du dépôt).
    """
    t = 0.0
    prev = villes[0]
    for ville in villes[1:]:
        t = max(t, ville.a) + ville.service
        t += _distance_minutes(prev, ville, scale)
        prev = ville
    t += _distance_minutes(prev, villes[0], scale)
    return t
 
 
# ---------------------------------------------------------------------------
# Fonction principale
# ---------------------------------------------------------------------------
 
def generer_dataset_tsptwd(
    n_clients: int,
    villes_raw: list[Ville],
    *,
    chemin_sortie: str | Path = "dataset_tsptwd.json",
    scale: float = 200.0,
    service_min: float = 5.0,
    service_max: float = 15.0,
    fenetre_largeur_min: float = 120.0,
    fenetre_largeur_max: float = 240.0,
    horizon: float = 1440.0,
    n_perturbations: int | None = None,
    alpha_min: float = 1.5,
    alpha_max: float = 3.5,
    seed: int = 42,
) -> dict:
    """
    Génère un jeu de données TSPTW-D à partir de coordonnées brutes.
 
    Paramètres
    ----------
    n_clients       : nombre de clients souhaités (hors dépôt).
    villes_raw      : liste de Ville (coordonnées brutes), len >= n_clients + 1.
    chemin_sortie   : chemin du fichier JSON à écrire.
    scale           : facteur de conversion distance → minutes.
    service_min/max : plage du temps de service s_i (minutes).
    fenetre_largeur_min/max : plage de la largeur [a_i, b_i].
    horizon         : fin de journée (b_0 du dépôt, en minutes).
    n_perturbations : nombre de perturbations. Par défaut ≈ n_clients // 5.
    alpha_min/max   : plage du facteur de perturbation alpha.
    seed            : graine aléatoire (reproductibilité).
 
    Contraintes respectées
    ----------------------
    1. a_i < b_i  pour tout nœud.
    2. Les fenêtres sont compatibles avec un ordre de visite naturel
       (on génère a_i en simulant la propagation, garantissant qu'au
       moins une permutation est réalisable).
    3. b_0 = horizon (journée complète du dépôt).
    4. Les perturbations ne dépassent pas l'horizon.
    5. c_ij(t) > 0 pour tout t (alpha > 0 assuré).
 
    Retourne
    --------
    Le dictionnaire représentant le jeu de données (aussi écrit en JSON).
    """
    rng = random.Random(seed)
 
    if len(villes_raw) < n_clients + 1:
        raise ValueError(
            f"villes_raw contient seulement {len(villes_raw)} villes "
            f"mais n_clients+1 = {n_clients + 1} sont nécessaires."
        )
 
    # --- 1. Sélection des nœuds ------------------------------------------
    echantillon = villes_raw[:n_clients + 1]   # dépôt = index 0
    depot_raw   = echantillon[0]
    clients_raw = echantillon[1:]
 
    # --- 2. Dépôt -----------------------------------------------------------
    depot = Ville(
        nom     = "Dépôt",
        x       = float(depot_raw.x),
        y       = float(depot_raw.y),
        fenetre = (0.0, float('inf')),  # dépôt toujours ouvert
        service = 0.0,
    )
 
    # --- 3. Fenêtres temporelles aléatoires pour les clients ---------------
    #
    # Principe : a_i est tiré uniformément sur [0, horizon - largeur_min]
    # de façon TOTALEMENT indépendante de l'ordre des villes et de la largeur.
    # La largeur est ensuite tirée séparément dans [largeur_min, largeur_max],
    # et clampée pour ne pas déborder de l'horizon.
    #
    # Résultat : des fenêtres qui se croisent, s'emboîtent ou s'excluent
    # sans aucun ordre, comme dans l'exemple :
    #   Ville 1 [10h-15h], Ville 2 [8h-16h], Ville 3 [11h-13h], ...
    #
    # Garanties :
    #   - b_i - a_i >= fenetre_largeur_min  (au moins largeur_min d'ouverture)
    #   - b_i <= horizon
    #   - a_i >= 0
    #
    clients: list[Ville] = []
 
    for idx, v_raw in enumerate(clients_raw, start=1):
        # Temps de service
        s_i = round(rng.uniform(service_min, service_max), 1)
 
        # 1) Tirer a_i librement sur toute la journée
        #    (on réserve juste fenetre_largeur_min à la fin pour b_i)
        a_i = round(rng.uniform(0.0, horizon - fenetre_largeur_min), 1)
 
        # 2) Tirer la largeur indépendamment, puis clamper à l'horizon
        largeur = round(rng.uniform(fenetre_largeur_min, fenetre_largeur_max), 1)
        b_i = min(round(a_i + largeur, 1), horizon)
 
        # 3) Garantir largeur minimale après clamp
        if b_i - a_i < fenetre_largeur_min:
            b_i = min(round(a_i + fenetre_largeur_min, 1), horizon)
 
        client = Ville(
            nom     = f"Ville {idx}",
            x       = float(v_raw.x),
            y       = float(v_raw.y),
            fenetre = (a_i, b_i),
            service = s_i,
        )
        clients.append(client)
 
    # --- 4. Perturbations ---------------------------------------------------
    if n_perturbations is None:
        n_perturbations = max(1, n_clients // 5)
 
    # Pool de tous les arcs possibles (indices 0..n_clients)
    tous_arcs = [
        (i, j)
        for i in range(n_clients + 1)
        for j in range(i + 1, n_clients + 1)
    ]
    arcs_choisis = rng.sample(tous_arcs, min(n_perturbations, len(tous_arcs)))
 
    perturbations = []
    for arc in arcs_choisis:
        t_start = round(rng.uniform(0, horizon * 0.6), 1)
        duree   = round(rng.uniform(30, horizon * 0.3), 1)
        t_end   = min(round(t_start + duree, 1), horizon)
        alpha   = round(rng.uniform(alpha_min, alpha_max), 2)
        perturbations.append({
            "arc":     list(arc),
            "t_start": t_start,
            "t_end":   t_end,
            "alpha":   alpha,
        })
 
    # --- 5. Construction du dictionnaire ------------------------------------
    dataset = {
        "meta": {
            "n_clients":   n_clients,
            "scale":       scale,
            "horizon":     horizon,
            "seed":        seed,
        },
        "depot": {
            "id":      0,
            "nom":     depot.nom,
            "x":       depot.x,
            "y":       depot.y,
            "a":       depot.a,
            "b":       None,          # None = toujours ouvert (inf)
            "service": depot.service,
        },
        "clients": [
            {
                "id":      i + 1,
                "nom":     c.nom,
                "x":       c.x,
                "y":       c.y,
                "a":       c.a,
                "b":       c.b,
                "service": c.service,
            }
            for i, c in enumerate(clients)
        ],
        "perturbations": perturbations,
    }
 
    # --- 6. Écriture du fichier JSON ----------------------------------------
    chemin_sortie = Path(chemin_sortie)
    chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
    with open(chemin_sortie, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
 
    print(f"[OK] Jeu de données généré : {chemin_sortie.resolve()}")
    print(f"     {n_clients} clients | {len(perturbations)} perturbation(s) | seed={seed}")
 
    return dataset
 
 
# ---------------------------------------------------------------------------
# Chargement d'un dataset depuis un fichier JSON
# ---------------------------------------------------------------------------
 
def charger_dataset_tsptwd(chemin: str | Path) -> tuple[list[Ville], list[dict]]:
    """
    Charge un fichier JSON généré par `generer_dataset_tsptwd`.
 
    Retourne
    --------
    villes       : liste[Ville] — index 0 = dépôt, 1..n = clients
    perturbations: list[dict]  — chaque dict : {arc, t_start, t_end, alpha}
    """
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)
 
    depot_d = data["depot"]
    depot = Ville(
        nom     = depot_d["nom"],
        x       = depot_d["x"],
        y       = depot_d["y"],
        fenetre = (depot_d["a"], depot_d["b"]),
        service = depot_d["service"],
    )
 
    clients = [
        Ville(
            nom     = c["nom"],
            x       = c["x"],
            y       = c["y"],
            fenetre = (c["a"], c["b"]),
            service = c["service"],
        )
        for c in data["clients"]
    ]
 
    perturbations = [
        {
            "arc":     tuple(p["arc"]),
            "t_start": p["t_start"],
            "t_end":   p["t_end"],
            "alpha":   p["alpha"],
        }
        for p in data["perturbations"]
    ]
 
    return [depot] + clients, perturbations


# ---------------------------------------------------------------------------
# Exemple d'utilisation
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Génération en lot — plusieurs tailles d'un coup
# ---------------------------------------------------------------------------

def _random_villes(n: int, seed: int = 0) -> list[Ville]:
    """Génère n villes aléatoires dans [0, 1]² (utilisé si villes_raw=None)."""
    rng = random.Random(seed)
    return [Ville(f"Raw {i}", rng.uniform(0, 1), rng.uniform(0, 1)) for i in range(n)]


def generer_batch_tsptwd(
    tailles: list[int],
    *,
    dossier_sortie: str | Path = "datasets",
    prefixe: str = "tsptwd_n",
    villes_raw: list[Ville] | None = None,
    perturbs_par_client: float = 0.1,
    seed: int = 42,
    **kwargs,
) -> dict[int, dict]:
    """
    Génère un fichier TSPTW-D pour chaque taille dans *tailles*.

    Paramètres
    ----------
    tailles             : liste de n_clients à générer, ex. [10, 50, 100, 200].
    dossier_sortie      : dossier de destination (créé si absent).
    prefixe             : préfixe du nom de fichier  → <prefixe><n>.json.
    villes_raw          : pool de villes brutes commun.  Si None, un pool
                          aléatoire de taille max(tailles)+1 est créé.
    perturbs_par_client : fraction de n utilisée pour n_perturbations
                          (ex. 0.1 → 10 % de n, minimum 1).
    seed                : graine globale (chaque taille dérive une sous-graine).
    **kwargs            : options transmises à generer_dataset_tsptwd()
                          (scale, horizon, service_min/max, …).

    Retourne
    --------
    dict {n: dataset_dict} pour chaque taille générée.
    """
    max_n = max(tailles)
    pool  = villes_raw if villes_raw is not None else _random_villes(max_n + 1, seed=seed)

    if len(pool) < max_n + 1:
        raise ValueError(
            f"villes_raw a {len(pool)} entrées mais la plus grande taille "            f"requiert {max_n + 1} nœuds (dépôt inclus)."        )

    resultats = {}
    for n in tailles:
        chemin = Path(dossier_sortie) / f"{prefixe}{n}.json"
        n_perturb = max(1, int(n * perturbs_par_client))
        sous_seed = seed + n          # seed reproductible par taille
        dataset = generer_dataset_tsptwd(
            n_clients       = n,
            villes_raw      = pool[:n + 1],
            chemin_sortie   = chemin,
            n_perturbations = n_perturb,
            seed            = sous_seed,
            **kwargs,
        )
        resultats[n] = dataset

    print(f"[BATCH TERMINÉ] {len(tailles)} fichier(s) dans '{dossier_sortie}/'")
    return resultats

    # --- Exemple 1 : petit jeu (20 clients) ---------------------------------
    villes_raw_10 = _fake_villes(15, seed=0)
    generer_dataset_tsptwd(
        n_clients    = 10,
        villes_raw   = villes_raw_10,
        chemin_sortie = "datasets/tsptwd_n10.json",
        seed         = 42,
    )

# ---------------------------------------------------------------------------
# Exemple d'utilisation
# ---------------------------------------------------------------------------
"""if __name__ == "__main__":
    generer_batch_tsptwd(
        tailles        = [5, 10, 50, 100, 200, 300, 500, 1000, 10000],
        dossier_sortie = "datasets",
        seed           = 42,
    )"""


'if __name__ == "__main__":\n    generer_batch_tsptwd(\n        tailles        = [5, 10, 50, 100, 200, 300, 500, 1000, 10000],\n        dossier_sortie = "datasets",\n        seed           = 42,\n    )'

In [9]:
"""
Chargement d'une instance TSPTW-D en dictionnaire
==================================================
Structure retournée — prête pour l'optimisation :

instance = {
    # --- méta ---
    "n"      : int,           # nombre de clients (hors dépôt)
    "scale"  : float,
    "horizon": float,

    # --- nœuds (index 0 = dépôt, 1..n = clients) ---
    "coords"  : {i: (x, y)},  # coordonnées
    "a"       : {i: float},    # ouverture de fenêtre
    "b"       : {i: float},    # fermeture de fenêtre
    "service" : {i: float},    # temps de service

    # --- coûts de base (distance euclidienne × scale) ---
    "c_base"  : {(i,j): float},  # symétrique, i ≠ j

    # --- perturbations ---
    "perturbations": [
        {"arc": (i,j), "t_start": float, "t_end": float, "alpha": float},
        ...
    ],
}

Fonctions exposées
------------------
charger_instance(chemin)        → dict  (chargement complet)
c_ij(instance, i, j, t)        → float (coût dynamique à l'instant t)
"""

import json
import math
from pathlib import Path


# ---------------------------------------------------------------------------
# Chargement principal
# ---------------------------------------------------------------------------

def charger_instance(chemin: str | Path) -> dict:
    """
    Charge un fichier JSON généré par `generer_dataset_tsptwd`
    et retourne un dictionnaire prêt pour l'optimisation.

    Paramètre
    ---------
    chemin : chemin vers le fichier .json

    Retourne
    --------
    instance : dict avec les clés décrites en en-tête de module.
    """
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)

    n       = data["meta"]["n_clients"]
    scale   = data["meta"]["scale"]
    horizon = data["meta"]["horizon"]

    # --- nœuds : dépôt (0) + clients (1..n) --------------------------------
    coords  = {}
    a       = {}
    b       = {}
    service = {}

    d = data["depot"]
    coords[0]  = (d["x"], d["y"])
    a[0]       = d["a"]
    b[0]       = float('inf') if d["b"] is None else d["b"]
    service[0] = d["service"]

    for c in data["clients"]:
        i          = c["id"]          # 1..n
        coords[i]  = (c["x"], c["y"])
        a[i]       = c["a"]
        b[i]       = c["b"]
        service[i] = c["service"]

    # --- coûts de base : distance euclidienne × scale -----------------------
    noeuds = list(range(n + 1))
    c_base = {
        (i, j): round(
            math.hypot(coords[i][0] - coords[j][0],
                       coords[i][1] - coords[j][1]) * scale,
            4,
        )
        for i in noeuds
        for j in noeuds
        if i != j
    }

    # --- perturbations ------------------------------------------------------
    perturbations = [
        {
            "arc":     tuple(p["arc"]),
            "t_start": p["t_start"],
            "t_end":   p["t_end"],
            "alpha":   p["alpha"],
        }
        for p in data.get("perturbations", [])
    ]

    instance = {
        "n":             n,
        "scale":         scale,
        "horizon":       horizon,
        "coords":        coords,
        "a":             a,
        "b":             b,
        "service":       service,
        "c_base":        c_base,
        "perturbations": perturbations,
    }

    return instance


# ---------------------------------------------------------------------------
# Coût dynamique (modèle TSPTW-D du Livrable 1)
# ---------------------------------------------------------------------------

def c_ij(instance: dict, i: int, j: int, t: float) -> float:
    """
    Coût de transit dynamique de i vers j en partant à l'instant t.

        c_ij(t) = c_base[i,j] × (1 + δ_ij(t))

    δ_ij(t) = alpha - 1  si une perturbation sur (i,j) ou (j,i) est active à t
    δ_ij(t) = 0          sinon

    Paramètres
    ----------
    instance : dict retourné par charger_instance()
    i, j     : indices des nœuds (0 = dépôt)
    t        : instant de départ (minutes)

    Retourne
    --------
    Coût en minutes (float > 0).
    """
    base = instance["c_base"][(i, j)]
    for p in instance["perturbations"]:
        arc = p["arc"]
        if arc == (i, j) or arc == (j, i):   # symétrique
            if p["t_start"] <= t <= p["t_end"]:
                return base * p["alpha"]
    return base


# ---------------------------------------------------------------------------
# Propagation temporelle (utilitaire pour les algorithmes)
# ---------------------------------------------------------------------------

def evaluer_tournee(instance: dict, permutation: list[int],
                    dynamique: bool = True) -> tuple[bool, float]:
    """
    Évalue une permutation de clients et retourne (réalisable, coût_total).

    La permutation ne contient que les indices clients (1..n).
    Le dépôt (0) est ajouté automatiquement en début et fin.

        τ_{k+1} = max(τ_k, a_k) + service_k + c_ij(t_départ)

    Paramètres
    ----------
    instance    : dict retourné par charger_instance()
    permutation : liste d'indices clients, ex. [3, 1, 4, 2]
    dynamique   : si True, utilise c_ij() (coûts avec perturbations)
                  si False, utilise c_base (coûts statiques)

    Retourne
    --------
    (True,  coût)        si la tournée est réalisable
    (False, float('inf')) sinon
    """
    a       = instance["a"]
    b       = instance["b"]
    service = instance["service"]
    c_base  = instance["c_base"]

    route = [0] + list(permutation) + [0]
    t     = 0.0
    cout  = 0.0

    for k in range(len(route) - 1):
        i, j = route[k], route[k + 1]

        # Fenêtre de départ du nœud i
        if t > b[i]:
            return False, float("inf")

        # Attente éventuelle + service
        t = max(t, a[i]) + service[i]

        # Transit
        transit = c_ij(instance, i, j, t) if dynamique else c_base[(i, j)]
        cout   += transit
        t      += transit

    # Vérification retour au dépôt (toujours ok si b[0] = inf)
    if b[0] != float('inf') and t > b[0]:
        return False, float("inf")

    return True, round(cout, 4)


# ---------------------------------------------------------------------------
# Affichage récapitulatif
# ---------------------------------------------------------------------------

def afficher_instance(instance: dict) -> None:
    """Affiche un résumé lisible de l'instance."""
    n = instance["n"]
    print(f"Instance TSPTW-D")
    print(f"  Clients       : {n}")
    print(f"  Horizon       : {instance['horizon']} min")
    print(f"  Scale         : {instance['scale']}")
    print(f"  Perturbations : {len(instance['perturbations'])}")
    print()
    print(f"  {'Nœud':<10} {'Coords':>20}  {'[a, b]':>20}  {'service':>8}")
    print("  " + "-" * 65)
    for i in range(n + 1):
        nom  = "Dépôt" if i == 0 else f"Client {i}"
        x, y = instance["coords"][i]
        ai, bi = instance["a"][i], instance["b"][i]
        bi_str = "∞" if bi == float('inf') else f"{bi:6.1f}"
        si = instance["service"][i]
        print(f"  {nom:<10} ({x:7.3f}, {y:7.3f})  [{ai:6.1f}, {bi_str:>6}]  {si:7.1f} min")

    if instance["perturbations"]:
        print()
        print(f"  {'Arc':<10} {'t_start':>10} {'t_end':>10} {'alpha':>8}")
        print("  " + "-" * 45)
        for p in instance["perturbations"]:
            print(f"  {str(p['arc']):<10} {p['t_start']:>10.1f} {p['t_end']:>10.1f} {p['alpha']:>8.2f}")


# ---------------------------------------------------------------------------
# Exemple d'utilisation
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Remplace le chemin par ton fichier généré
    instance = charger_instance("datasets/tsptwd_n10.json")

    afficher_instance(instance)

    # Accès direct aux structures utiles pour un algo
    n      = instance["n"]
    a      = instance["a"]       # ouvertures
    b      = instance["b"]       # fermetures
    s      = instance["service"] # temps de service
    c_base = instance["c_base"]  # {(i,j): float}

    # Coût dynamique entre nœuds 1 et 2 à t=50 min
    print(f"\nc(1,2) à t=50 min : {c_ij(instance, 1, 2, 50):.2f} min")

    # Évaluation d'une tournée exemple
    perm = list(range(1, n + 1))   # ordre naturel
    ok, cout = evaluer_tournee(instance, perm, dynamique=True)
    print(f"Tournée ordre naturel — réalisable: {ok}, coût: {cout:.1f} min")

Instance TSPTW-D
  Clients       : 10
  Horizon       : 1440.0 min
  Scale         : 200.0
  Perturbations : 2

  Nœud                     Coords                [a, b]   service
  -----------------------------------------------------------------
  Dépôt      (  0.844,   0.758)  [   0.0,      ∞]      0.0 min
  Client 1   (  0.421,   0.259)  [  33.0,  186.0]     11.4 min
  Client 2   (  0.511,   0.405)  [ 972.1, 1173.3]      7.2 min
  Client 3   (  0.784,   0.303)  [ 114.8,  285.4]     13.9 min
  Client 4   (  0.477,   0.583)  [ 288.6,  469.2]      5.3 min
  Client 5   (  0.908,   0.505)  [ 262.5,  460.5]      5.3 min
  Client 6   (  0.282,   0.756)  [ 291.0,  481.7]     10.4 min
  Client 7   (  0.618,   0.251)  [   8.6,  225.3]     13.1 min
  Client 8   (  0.910,   0.983)  [ 449.1,  587.8]     12.0 min
  Client 9   (  0.810,   0.902)  [ 444.3,  575.4]     14.6 min
  Client 10  (  0.310,   0.730)  [1118.7, 1311.1]      6.0 min

  Arc           t_start      t_end    alpha
  --------------

In [10]:
def mst(instance: dict, dynamique: bool = False, t_ref: float = 0.0) -> tuple[list[tuple[int,int,float]], float]:
      """
      Calcule le MST (Kruskal) sur l'instance TSPTW-D.

      Paramètres
      ----------
      instance  : dict retourné par charger_instance()
      dynamique : si True, utilise c_ij() à t_ref au lieu de c_base
      t_ref     : instant de référence pour les coûts dynamiques

      Retourne
      --------
      (aretes, cout_total)
      aretes : liste de (i, j, poids) triée par poids croissant
      """
      n = instance["n"]
      noeuds = list(range(n + 1))

      # Collecte et tri des arcs (non orienté → i < j seulement)
      arcs = []
      for i in noeuds:
          for j in noeuds:
              if i < j:
                  if dynamique:
                      # coût moyen des deux sens
                      w = (c_ij(instance, i, j, t_ref) + c_ij(instance, j, i, t_ref)) / 2
                  else:
                      w = instance["c_base"][(i, j)]
                  arcs.append((w, i, j))
      arcs.sort()

      # Union-Find
      parent = list(range(n + 1))
      rank   = [0] * (n + 1)

      def find(x):
          while parent[x] != x:
              parent[x] = parent[parent[x]]
              x = parent[x]
          return x

      def union(x, y):
          rx, ry = find(x), find(y)
          if rx == ry:
              return False
          if rank[rx] < rank[ry]:
              rx, ry = ry, rx
          parent[ry] = rx
          if rank[rx] == rank[ry]:
              rank[rx] += 1
          return True

      aretes_mst = []
      cout_total = 0.0
      for w, i, j in arcs:
          if union(i, j):
              aretes_mst.append((i, j, w))
              cout_total += w
              if len(aretes_mst) == n:   # MST complet
                  break

      return aretes_mst, round(cout_total, 4)

In [ ]:
def _tw_arc_feasible(instance: dict, i: int, j: int) -> bool:
    """
    True si l'arc {i,j} est utilisable dans AU MOINS une direction,
    en partant le plus tôt possible.

    Direction i→j : a[i] + service[i] + c_base[i,j]  ≤  b[j]
    Direction j→i : a[j] + service[j] + c_base[j,i]  ≤  b[i]

    Un arc infaisable dans les DEUX sens ne peut appartenir à
    aucune tournée réalisable → exclu du 1-arbre (coût inf).

    Note : on utilise c_base pour le test (coût minimum possible).
    Si même c_base ne permet pas d'atteindre j avant b[j],
    aucun coût plus élevé n'y parvient non plus.
    """
    a = instance["a"]
    b = instance["b"]
    s = instance["service"]
    c = instance["c_base"]
    ok_ij = (a[i] + s[i] + c[(i, j)]) <= b[j]
    ok_ji = (a[j] + s[j] + c[(j, i)]) <= b[i]
    return ok_ij or ok_ji


def c_perturb(instance: dict, i: int, j: int,
              mode: str = "expected",
              t_ref: float = 0.0,
              use_tw: bool = False) -> float:
    """
    Coût d'arc (i, j) pour la construction du 1-arbre,
    selon le mode de traitement des perturbations,
    avec option de filtrage par fenêtres temporelles.

    Modes
    -----
    static     : c_base[i,j]  — ignore toutes les perturbations
    snapshot   : c_ij(t_ref)  — perturbation active à l'instant t_ref uniquement
    worst_case : c_base * alpha  si une perturbation couvre (i,j), sinon c_base
    expected   : c_base * (1 + (alpha-1) * fraction_active)
                 fraction_active = durée_perturbation / horizon

    use_tw
    ------
    Si True : retourne float('inf') si l'arc est TW-infaisable dans les
              deux directions. L'arc est alors exclu du MST / 1-arbre,
              ce qui rend la borne plus serrée tout en restant valide.

              On N'ajoute PAS les temps d'attente au coût — cela
              convertirait une borne sur le coût de transit en une borne
              sur le makespan (métrique différente), potentiellement
              supérieure à l'optimal réel.
    """
    if use_tw and not _tw_arc_feasible(instance, i, j):
        return float("inf")

    base    = instance["c_base"][(i, j)]
    horizon = instance["horizon"]

    if mode == "static":
        return base

    if mode == "snapshot":
        return c_ij(instance, i, j, t_ref)

    for p in instance["perturbations"]:
        arc = p["arc"]
        if arc == (i, j) or arc == (j, i):
            alpha   = p["alpha"]
            t_start = p["t_start"]
            t_end   = p["t_end"]

            if mode == "worst_case":
                return base * alpha

            if mode == "expected":
                frac = max(0.0, min(t_end, horizon) - max(0.0, t_start)) / horizon
                return base * (1.0 + (alpha - 1.0) * frac)

    return base


def one_tree(instance: dict,
             mode: str = "expected",
             t_ref: float = 0.0,
             use_tw: bool = False) -> tuple[list[tuple[int, int, float]], float]:
    """
    Calcule un 1-arbre (borne inférieure de Held-Karp) pour le TSPTW-D.

    Algorithme
    ----------
    1. Retire le dépôt (nœud 0).
    2. Construit un MST de Kruskal sur les nœuds clients {1..n}
       avec les coûts donnés par c_perturb(mode, use_tw).
    3. Ajoute les 2 arêtes de coût minimal reliant le dépôt au reste.

    Paramètres
    ----------
    instance : dict retourné par charger_instance()
    mode     : 'static' | 'snapshot' | 'worst_case' | 'expected'
    t_ref    : instant de référence (mode='snapshot' uniquement)
    use_tw   : si True, exclut les arcs TW-infaisables dans les deux
               directions (borne plus serrée, toujours valide)

    Retourne
    --------
    (aretes, cout_total)
    aretes     : liste de (i, j, poids) — MST sur {1..n} + 2 arêtes dépôt
    cout_total : borne inférieure sur le coût de transit (minutes)
    """
    n       = instance["n"]
    clients = list(range(1, n + 1))

    # -- 1. MST de Kruskal sur {1..n} ---------------------------------------
    arcs = sorted(
        (c_perturb(instance, i, j, mode, t_ref, use_tw), i, j)
        for i in clients
        for j in clients
        if i < j
    )

    parent = list(range(n + 1))
    rank   = [0] * (n + 1)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx == ry:
            return False
        if rank[rx] < rank[ry]:
            rx, ry = ry, rx
        parent[ry] = rx
        if rank[rx] == rank[ry]:
            rank[rx] += 1
        return True

    mst_aretes = []
    mst_cout   = 0.0
    for w, i, j in arcs:
        if w == float("inf"):
            continue
        if union(i, j):
            mst_aretes.append((i, j, w))
            mst_cout += w
            if len(mst_aretes) == n - 1:
                break

    # -- 2. 2 arêtes les moins chères depuis le dépôt (nœud 0) -------------
    depot_arcs = sorted(
        (c_perturb(instance, 0, j, mode, t_ref, use_tw), j)
        for j in clients
    )
    depot_feasible = [(w, j) for w, j in depot_arcs if w < float("inf")]
    if len(depot_feasible) < 2:
        raise ValueError(
            "Moins de 2 arcs faisables depuis le dépôt — "
            "instance trop contrainte pour un 1-arbre avec use_tw=True."
        )
    w1, j1 = depot_feasible[0]
    w2, j2 = depot_feasible[1]

    aretes = mst_aretes + [(0, j1, w1), (0, j2, w2)]
    cout   = round(mst_cout + w1 + w2, 4)

    return aretes, cout


In [16]:
instance = charger_instance("datasets/tsptwd_n500.json")

print(f"Instance : {instance['n']} clients, {len(instance['perturbations'])} perturbation(s)")
print(f"  {'Mode':<15} {'Coût 1-arbre':>14}  Commentaire")
print("  " + "-" * 60)

infos = {
    "static"    : "borne inférieure vraie  (pas de perturbation)",
    "expected"  : "borne réaliste pondérée (fraction du temps perturbé)",
    "snapshot"  : "coûts à t=0 min",
    "worst_case": "majoration (toutes perturbations actives)",
}
for mode, label in infos.items():
    aretes, cout = one_tree(instance, mode=mode)
    print(f"  {mode:<15} {cout:>14.4f}  {label}")

# Détail du mode expected
print()
aretes, cout = one_tree(instance, mode="expected", use_tw=True)
print(f"1-arbre (expected) — {len(aretes)} arêtes, borne = {cout} min")
print(f"  {'Arc':<12} {'Poids':>10}  Type")
print("  " + "-" * 35)
for i, j, w in sorted(aretes, key=lambda x: x[2]):
    tag = "depot" if i == 0 or j == 0 else "mst"
    print(f"  {i:>2} — {j:<5}   {w:>9.4f}  {tag}")


Instance : 500 clients, 50 perturbation(s)
  Mode              Coût 1-arbre  Commentaire
  ------------------------------------------------------------
  static               2975.2574  borne inférieure vraie  (pas de perturbation)
  expected             2975.2574  borne réaliste pondérée (fraction du temps perturbé)
  snapshot             2975.2574  coûts à t=0 min
  worst_case           2975.2574  majoration (toutes perturbations actives)

1-arbre (expected) — 501 arêtes, borne = 6054.4659 min
  Arc               Poids  Type
  -----------------------------------
  292 — 461        0.7199  mst
  360 — 393        0.9584  mst
  184 — 432        1.0292  mst
  62 — 75         1.0575  mst
  47 — 436        1.2874  mst
  258 — 317        1.4485  mst
  230 — 435        1.4879  mst
  145 — 475        1.6390  mst
  134 — 231        1.7575  mst
  46 — 120        1.8355  mst
  49 — 476        2.1392  mst
  147 — 406        2.1573  mst
  195 — 380        2.1903  mst
  141 — 489        2.3520  mst

In [8]:
instance = charger_instance("datasets/tsptwd_n10.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

Coût MST : 349.8983
  6 — 10  :  7.68
  8 — 9  :  25.62
  0 — 9  :  29.64
  1 — 2  :  34.38
  3 — 7  :  34.73
  2 — 4  :  36.36
  2 — 7  :  37.59
  4 — 10  :  44.34
  3 — 5  :  47.33
  0 — 5  :  52.23


In [ ]:
instance = charger_instance("datasets/tsptwd_n50.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
instance = charger_instance("datasets/tsptwd_n100.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
instance = charger_instance("datasets/tsptwd_n200.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
instance = charger_instance("datasets/tsptwd_n300.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
instance = charger_instance("datasets/tsptwd_n500.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
instance = charger_instance("datasets/tsptwd_n1000.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
instance = charger_instance("datasets/tsptwd_n10000.json")

# MST statique (c_base)
aretes, cout = mst(instance)

# MST dynamique à t=0
aretes, cout = mst(instance, dynamique=True, t_ref=0.0)

print(f"Coût MST : {cout}")
for i, j, w in aretes:
    print(f"  {i} — {j}  :  {w:.2f}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
 
 
def afficher_mst(
    aretes: list[tuple[int, int, float]],
    titre: str = "Arbre couvrant minimal (MST)",
    layout: str = "spring",       # "spring" | "kamada" | "spectral"
    figsize: tuple = (12, 8),
    palette: dict | None = None,
) -> None:
    """
    Affiche graphiquement le MST retourné par la fonction mst().
 
    Paramètres
    ----------
    aretes  : liste de (i, j, poids) — sortie directe de mst()
    titre   : titre de la figure
    layout  : algorithme de placement des noeuds
    figsize : taille de la figure matplotlib
    palette : dict optionnel pour personnaliser les couleurs
    """
    couleurs = {
        "node_color":  "#AFA9EC",
        "depot_color": "#5DCAA5",
        "edge_color":  "#3C3489",
        "font_color":  "#26215C",
        "label_color": "#3C3489",
    }
    if palette:
        couleurs.update(palette)
 
    # Coût calculé directement depuis la liste
    cout_total = round(sum(w for _, _, w in aretes), 4)
 
    # Construction du graphe
    G = nx.Graph()
    for i, j, w in aretes:
        G.add_edge(i, j, weight=w)
 
    noeuds = sorted(G.nodes())
 
    layouts = {
        "spring":   lambda g: nx.spring_layout(g, seed=42, k=2.5),
        "kamada":   nx.kamada_kawai_layout,
        "spectral": nx.spectral_layout,
    }
    pos = layouts.get(layout, layouts["spring"])(G)
 
    node_colors = [
        couleurs["depot_color"] if n == 0 else couleurs["node_color"]
        for n in noeuds
    ]
 
    poids = [G[u][v]["weight"] for u, v in G.edges()]
    w_min, w_max = min(poids), max(poids)
    span = (w_max - w_min) or 1
    widths = [1.0 + 3.5 * (w - w_min) / span for w in poids]
 
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor("#F8F7FF")
    fig.patch.set_facecolor("#F8F7FF")
 
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        nodelist=noeuds,
        node_color=node_colors,
        node_size=700,
        edgecolors=couleurs["edge_color"],
        linewidths=1.5,
    )
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        font_size=11,
        font_color=couleurs["font_color"],
        font_weight="bold",
    )
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edge_color=couleurs["edge_color"],
        width=widths,
        alpha=0.75,
    )
 
    edge_labels = {(u, v): f"{G[u][v]['weight']:.2f}" for u, v in G.edges()}
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels, ax=ax,
        font_size=8,
        font_color=couleurs["label_color"],
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7),
    )
 
    legende = [
        mpatches.Patch(color=couleurs["depot_color"], label="Dépôt (noeud 0)"),
        mpatches.Patch(color=couleurs["node_color"],  label="Client"),
    ]
    ax.legend(handles=legende, loc="upper left", fontsize=10, framealpha=0.8)
 
    ax.set_title(titre, fontsize=14, pad=16)
 
    # Encadré coût total centré en bas de la figure
    ax.text(
        0.5, -0.04,
        f"Coût total  :  {cout_total}",
        transform=ax.transAxes,
        ha="center", va="top",
        fontsize=13, fontweight="bold",
        color=couleurs["edge_color"],
        bbox=dict(
            boxstyle="round,pad=0.5",
            fc="#EEEDFE",          # c-purple 50
            ec=couleurs["edge_color"],
            linewidth=1.2,
        ),
    )
    ax.text(
        0.98, 0.02,
        f"{len(noeuds)} noeuds · {len(aretes)} arêtes",
        transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9,
        color="gray",
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:

cout_exemple = sum(w for _, _, w in aretes)

afficher_mst(aretes, round(cout_exemple, 4))